# Hello LangGraph

가장 기본적인 2단계 순차 그래프 예제입니다.

```
START → GREETING(인사말 생성) → PROCESSING(최종 메시지 생성) → END
```

| 개념 | 설명 |
|---|---|
| State | 그래프 전체가 공유하는 데이터 (Pydantic 모델) |
| Node | 상태를 입력받아 **변경할 값만** dict로 반환하는 함수 |
| Edge | 노드 간 실행 순서 |
| Compile | 그래프를 실행 가능한 앱으로 변환 |

**사전 준비:** `uv add langgraph pydantic ipykernel` (또는 `poetry add ...`) 후 커널을 `.venv`로 선택하세요.

## 1. 라이브러리 임포트

In [ ]:
from typing import Dict, Any                         # 노드 반환 타입 힌트
from langgraph.graph import StateGraph, START, END   # 그래프 빌더, 시작/종료 지점
from pydantic import BaseModel, Field                # 상태 스키마 정의
from IPython.display import Image, display           # 노트북에서 이미지 표시

## 2. 워크플로우 단계(노드 이름) 정의
노드 이름을 상수로 관리하면 오타를 방지할 수 있습니다.

In [ ]:
class WorkflowStep:
    GREETING = "GREETING"       # 인사말 생성 노드
    PROCESSING = "PROCESSING"   # 메시지 처리 노드

## 3. 그래프 상태 정의
모든 노드가 읽고 쓰는 공유 데이터입니다. 노드가 반환하는 dict의 키는 **이 필드 이름과 정확히 같아야** 상태에 반영됩니다.

In [ ]:
class GraphState(BaseModel):
    name: str = Field(default="", description="사용자 이름")                    # 입력값
    greeting: str = Field(default="", description="생성된 인사말")              # GREETING 노드 결과
    processed_message: str = Field(default="", description="처리된 최종 메시지")  # PROCESSING 노드 결과

## 4. 노드 함수 정의

In [ ]:
# 첫 번째 노드: 이름으로 인사말 생성
def generate_greeting(state: GraphState) -> Dict[str, Any]:
    name = state.name or "아무개"          # 이름이 없으면 기본값
    greeting = f"안녕하세요, {name}님!"
    print(f"[generate_greeting] 인사말 생성: {greeting}")
    return {"greeting": greeting}         # 변경할 필드만 반환


# 두 번째 노드: 인사말에 환영 문구를 붙여 최종 메시지 생성
def process_message(state: GraphState) -> Dict[str, Any]:
    processed_message = f"{state.greeting} LangGraph에 오신 것을 환영합니다!"
    print(f"[process_message] 최종 메시지: {processed_message}")
    return {"processed_message": processed_message}

## 5. 그래프 생성 및 컴파일
※ 위 셀(상태·노드)을 수정했다면 이 셀도 다시 실행해야 반영됩니다.

In [ ]:
workflow = StateGraph(GraphState)                     # 상태 스키마 기반 그래프 빌더

# 노드 등록
workflow.add_node(WorkflowStep.GREETING, generate_greeting)
workflow.add_node(WorkflowStep.PROCESSING, process_message)

# 엣지 연결: START → GREETING → PROCESSING → END
workflow.add_edge(START, WorkflowStep.GREETING)
workflow.add_edge(WorkflowStep.GREETING, WorkflowStep.PROCESSING)
workflow.add_edge(WorkflowStep.PROCESSING, END)

app = workflow.compile()                              # 실행 가능한 앱으로 변환

## 6. 그래프 구조 시각화
Mermaid 이미지를 셀 아래에 바로 표시합니다. (mermaid.ink 웹 API 사용 → 인터넷 연결 필요)

In [ ]:
display(Image(app.get_graph().draw_mermaid_png()))

## 7. 그래프 실행
`name` 값을 바꿔서 실행해 보세요.

In [ ]:
initial_state = GraphState(name="홍길동")
print("초기 상태:", initial_state.model_dump())
print("\n--- 그래프 실행 시작 ---")

final_state = app.invoke(initial_state)               # 반환값은 dict

print("--- 그래프 실행 종료 ---\n")
print(f"결과 메시지: {final_state['processed_message']}")

## 8. 최종 상태 확인
셀 마지막 줄에 변수명만 쓰면 자동으로 출력됩니다.

In [ ]:
final_state